# Part 2: Numerical Experiments in Cloud Microphysics
## Scientific Exploration with PyLCM

### Structure
Each experiment follows the scientific method:
1. **Hypothesis** — predict the outcome before running
2. **Setup** — define simulation parameters
3. **Run** — execute simulations
4. **Analysis** — visualize and quantify results
5. **Discussion** — interpret findings, connect to theory

### Prerequisites
- Complete Part 1: Cloud Microphysics Foundations
- Familiarity with `run_simulation()` and `plot_comparison()` functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm
import copy, warnings, time
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '.')

%matplotlib inline

from PyLCM.parameters import *
from PyLCM.micro_particle import *
from PyLCM.aero_init import aero_init, lognormal_pdf, r_equi
from PyLCM.parcel import ascend_parcel, parcel_rho, create_env_profiles
from PyLCM.condensation import esatw, sigma_air_liq, radius_liquid_euler, drop_condensation, compute_tau_phase
from PyLCM.collision import collection, ws_drops_beard, ws_drops_stokes, E_H80, E_S09, gck, E_turb
from Post_process.analysis import ts_analysis
from PyLCM.widget import AEROSOL_PRESETS

In [ ]:
def run_simulation(params, seed=42):
    """Run a PyLCM simulation programmatically.
    
    Parameters
    ----------
    params : dict with keys:
        label, T0 (K), P0 (Pa), RH (0-1), w (m/s),
        n_ptcl (int), nt (int), dt (float),
        N (list, cm⁻³), mu (list, µm), sigma (list, geometric std), kappa (list),
        do_condensation (bool), do_collision (bool),
        switch_kelvin (bool), switch_solute (bool),
        switch_E_constant (bool), switch_vt_simple (bool),
        switch_turb_kernel (bool), epsilon_turb (float)
    seed : int
    
    Returns dict with: label, time, T, z, P, RH, qv, qc, qr, qa, nc, nr, na, 
                       spectra, rc_avg, con, evp, acc, aut
    """
    p = {  # defaults
        'label': 'Run', 'T0': 293.2, 'P0': 101300.0, 'RH': 0.88, 'w': 1.0,
        'n_ptcl': 1000, 'nt': 1200, 'dt': 1.0,
        'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
        'do_condensation': True, 'do_collision': False,
        'switch_kelvin': True, 'switch_solute': True,
        'switch_E_constant': False, 'switch_vt_simple': False,
        'switch_turb_kernel': False, 'epsilon_turb': 0.0,
        'switch_kappa_koehler': True,
    }
    p.update(params)
    
    T0, P0, RH, w = p['T0'], p['P0'], p['RH'], p['w']
    nt, dt, n_ptcl = p['nt'], p['dt'], p['n_ptcl']
    z0, zmax = 0.0, 3000.0
    
    # Convert aerosol parameters - CRITICAL unit conversion
    N_raw = np.array(p['N'])
    mu_raw = np.array(p['mu'])
    sigma_raw = np.array(p['sigma'])
    kappa_list = p['kappa']
    
    # Remove zero-N modes
    mask = N_raw > 0
    N_raw = N_raw[mask]
    mu_raw = mu_raw[mask]
    sigma_raw = sigma_raw[mask]
    
    N_aero = N_raw * 1e6              # cm⁻³ → m⁻³
    mu_aero = np.log(mu_raw * 1e-6)   # µm → m → ln(m)
    sigma_aero = np.log(sigma_raw)     # geometric std → ln(σ)
    
    # Environmental profiles
    theta_init = T0 * (p0 / P0)**(r_a / cp)
    theta_profiles = theta_init + 5e-3 * z_env
    
    # Initial moisture
    e_s0 = esatw(T0)
    q0 = RH * e_s0 / (P0 - RH * e_s0) * r_a / rv
    
    # Initialize particles
    np.random.seed(seed)
    T_parcel, q_parcel, particles_list = aero_init(
        'Random', n_ptcl, P0, z0, T0, q0,
        N_aero, mu_aero, sigma_aero, rho_aero, kappa_list, p['switch_kappa_koehler'])
    
    P_parcel, z_parcel = P0, z0
    S_lst = 0.0
    
    # Output arrays
    time_arr = np.zeros(nt + 1)
    T_arr = np.zeros(nt + 1); z_arr = np.zeros(nt + 1); P_arr = np.zeros(nt + 1)
    RH_arr = np.zeros(nt + 1); qv_arr = np.zeros(nt + 1)
    qc_arr = np.zeros(nt + 1); qr_arr = np.zeros(nt + 1); qa_arr = np.zeros(nt + 1)
    nc_arr = np.zeros(nt + 1); nr_arr = np.zeros(nt + 1); na_arr = np.zeros(nt + 1)
    con_arr = np.zeros(nt + 1); evp_arr = np.zeros(nt + 1)
    acc_arr = np.zeros(nt + 1); aut_arr = np.zeros(nt + 1)
    rc_avg_arr = np.zeros(nt + 1)
    spectra_all = np.zeros((nt + 1, len(rm_spec)))
    
    # Initial analysis
    rho_p, V_p, air_mass = parcel_rho(P_parcel, T_parcel)
    sp, qa, qc, qr, na, nc, nr, _, rc_avg, _ = ts_analysis(
        particles_list, air_mass, rm_spec, n_bins, n_ptcl)
    T_arr[0] = T_parcel; z_arr[0] = z_parcel; P_arr[0] = P_parcel
    qv_arr[0] = q_parcel; qc_arr[0] = qc; qr_arr[0] = qr; qa_arr[0] = qa
    nc_arr[0] = nc; nr_arr[0] = nr; na_arr[0] = na
    rc_avg_arr[0] = rc_avg; spectra_all[0] = sp
    RH_arr[0] = (q_parcel * P_parcel / (q_parcel + r_a / rv)) / esatw(T_parcel)
    
    for t in range(nt):
        tt = (t + 1) * dt
        time_arr[t + 1] = tt
        
        z_parcel, T_parcel, P_parcel = ascend_parcel(
            z_parcel, T_parcel, P_parcel, w, dt, tt, zmax, theta_profiles)
        rho_p, V_p, air_mass = parcel_rho(P_parcel, T_parcel)
        
        ct = at = et = da = 0.0
        ac = au = pr = 0.0
        
        if p['do_condensation']:
            particles_list, T_parcel, q_parcel, S_lst, ct, at, et, da = drop_condensation(
                particles_list, T_parcel, q_parcel, P_parcel, nt, dt, air_mass,
                S_lst, rho_aero, False, ct, at, et, da,
                p['switch_kappa_koehler'], p['switch_kelvin'], p['switch_solute'])
            con_arr[t+1] = 1e3 * ct / air_mass / dt
            evp_arr[t+1] = 1e3 * et / air_mass / dt
        
        if p['do_collision']:
            particles_list, ac, au, pr = collection(
                dt, particles_list, rho_p, rho_liq, P_parcel, T_parcel,
                ac, au, pr, False, z_parcel, zmax, w,
                p['switch_E_constant'], p['switch_vt_simple'],
                p['switch_turb_kernel'], p['epsilon_turb'])
            acc_arr[t+1] = 1e3 * ac / air_mass / dt
            aut_arr[t+1] = 1e3 * au / air_mass / dt
        
        sp, qa, qc, qr, na, nc, nr, _, rc_avg, _ = ts_analysis(
            particles_list, air_mass, rm_spec, n_bins, len(particles_list))
        
        T_arr[t+1] = T_parcel; z_arr[t+1] = z_parcel; P_arr[t+1] = P_parcel
        qv_arr[t+1] = q_parcel
        qc_arr[t+1] = qc; qr_arr[t+1] = qr; qa_arr[t+1] = qa
        nc_arr[t+1] = nc; nr_arr[t+1] = nr; na_arr[t+1] = na
        rc_avg_arr[t+1] = rc_avg; spectra_all[t+1] = sp
        RH_arr[t+1] = (q_parcel * P_parcel / (q_parcel + r_a / rv)) / esatw(T_parcel)
    
    return {
        'label': p['label'], 'time': time_arr, 'T': T_arr, 'z': z_arr, 'P': P_arr,
        'RH': RH_arr, 'qv': qv_arr, 'qc': qc_arr, 'qr': qr_arr, 'qa': qa_arr,
        'nc': nc_arr, 'nr': nr_arr, 'na': na_arr, 'spectra': spectra_all,
        'rc_avg': rc_avg_arr, 'con': con_arr, 'evp': evp_arr,
        'acc': acc_arr, 'aut': aut_arr,
    }


def plot_comparison(results_list, title='Comparison'):
    """Overlay 1-6 simulation results on a 2x4 panel grid."""
    fig, axs = plt.subplots(2, 4, figsize=(20, 9))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    colors = [plt.cm.tab10(i) for i in range(len(results_list))]
    
    for i, r in enumerate(results_list):
        c = colors[i]
        t = r['time']
        lbl = r['label']
        
        axs[0,0].plot(t, r['T'] - 273.15, color=c, label=lbl)
        axs[0,0].set_ylabel('T (°C)'); axs[0,0].set_xlabel('Time (s)')
        
        axs[0,1].plot(t, r['RH'] * 100, color=c, label=lbl)
        axs[0,1].axhline(100, color='gray', ls='--', lw=0.5)
        axs[0,1].set_ylabel('RH (%)'); axs[0,1].set_xlabel('Time (s)')
        
        axs[0,2].plot(t, r['qc'], color=c, ls='-', label=f"{lbl} qc")
        axs[0,2].plot(t, r['qr'], color=c, ls='--', label=f"{lbl} qr")
        axs[0,2].set_ylabel('q (g/kg)'); axs[0,2].set_xlabel('Time (s)')
        
        axs[0,3].plot(t, r['nc'], color=c, ls='-', label=f"{lbl} Nc")
        axs[0,3].plot(t, r['nr'], color=c, ls='--', label=f"{lbl} Nr")
        axs[0,3].set_ylabel('N (cm⁻³)'); axs[0,3].set_xlabel('Time (s)')
        
        nt_r = len(t) - 1
        t_mid = nt_r // 2
        spec_final = copy.deepcopy(r['spectra'][nt_r])
        spec_mid = copy.deepcopy(r['spectra'][t_mid])
        
        axs[1,0].plot(rm_spec * 1e6, spec_final / 1e6, color=c, label=lbl)
        axs[1,0].set_xscale('log'); axs[1,0].set_xlabel('Radius (µm)')
        axs[1,0].set_ylabel('dN/dlogr (cm⁻³)'); axs[1,0].set_title('DSD at t_final')
        
        axs[1,1].plot(rm_spec * 1e6, spec_mid / 1e6, color=c, label=lbl)
        axs[1,1].set_xscale('log'); axs[1,1].set_xlabel('Radius (µm)')
        axs[1,1].set_ylabel('dN/dlogr (cm⁻³)'); axs[1,1].set_title('DSD at t_mid')
        
        axs[1,2].plot(t, r['qc'] + r['qr'], color=c, label=lbl)
        axs[1,2].set_ylabel('LWC (g/kg)'); axs[1,2].set_xlabel('Time (s)')
        axs[1,2].set_title('Total LWC')
        
        window = min(60, max(1, nt_r // 20))
        if np.any(r['aut'] > 0) or np.any(r['acc'] > 0):
            aut_smooth = np.convolve(r['aut'], np.ones(window)/window, mode='same')
            acc_smooth = np.convolve(r['acc'], np.ones(window)/window, mode='same')
            axs[1,3].plot(t, aut_smooth, color=c, ls='-', label=f"{lbl} aut")
            axs[1,3].plot(t, acc_smooth, color=c, ls='--', label=f"{lbl} acc")
        axs[1,3].set_ylabel('Rate (g/kg/s)'); axs[1,3].set_xlabel('Time (s)')
        axs[1,3].set_title('Autoconv / Accretion')
    
    for ax in axs.flat:
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    fig.tight_layout(); plt.show()

### Default Parameters
- Part 2 experiments use `n_ptcl=1000, nt=2400` by default (~40s wall clock)
- All simulations use Maritime preset as baseline unless otherwise noted
- Random seed = 42 for reproducibility (varied in Experiment 4)

## Experiment 1: Maritime vs Continental vs Arctic

### Background
Cloud properties vary dramatically depending on the aerosol environment:
- **Maritime**: few, large, hygroscopic aerosols over oceans
- **Continental**: many, small, less hygroscopic aerosols over land
- **Arctic**: ultra-clean conditions with very few CCN

### Hypothesis
Before running the simulations, write your predictions:
1. Which environment produces rain fastest? Why?
2. Which has the highest peak cloud water content ($q_c$)?
3. Which produces the narrowest drop size distribution?

In [ ]:
# Experiment 1: Three aerosol environments
presets = ['Maritime', 'Continental', 'Arctic']
exp1_params = []

for name in presets:
    p = AEROSOL_PRESETS[name]
    exp1_params.append({
        'label': name,
        'n_ptcl': 1000, 'nt': 2400,
        'N': p['N'], 'mu': p['mu'], 'sigma': p['sigma'], 'kappa': p['kappa'],
        'do_condensation': True, 'do_collision': True,
    })

# Print summary
for ep in exp1_params:
    N_total = sum(n for n in ep['N'] if n > 0)
    print(f"{ep['label']:12s}: N_total = {N_total:7.1f} cm⁻³")

In [ ]:
%%time
exp1_results = []
for params in exp1_params:
    print(f"Running {params['label']}...")
    exp1_results.append(run_simulation(params))
print("All simulations complete!")

In [ ]:
plot_comparison(exp1_results, 'Experiment 1: Maritime vs Continental vs Arctic')

In [ ]:
# DSD snapshots at key times
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
snap_times = [600, 1200, 2400]
colors_e1 = [plt.cm.tab10(i) for i in range(3)]

for ax, t_snap in zip(axes, snap_times):
    for r, col in zip(exp1_results, colors_e1):
        idx = min(t_snap, len(r['time']) - 1)
        ax.plot(rm_spec * 1e6, r['spectra'][idx] / 1e6, color=col, lw=2, label=r['label'])
    ax.set_xscale('log')
    ax.set_xlabel('Radius (µm)'); ax.set_ylabel('dN/dlogr (cm⁻³)')
    ax.set_title(f't = {t_snap} s')
    ax.axvline(25, color='gray', ls='--', alpha=0.5, label='Rain threshold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

fig.suptitle('DSD Evolution: Maritime vs Continental vs Arctic', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Results summary table
print(f"{'Preset':12s} {'N_c max':>10s} {'q_c max':>10s} {'q_r final':>10s} {'Rain onset':>12s}")
print("-" * 60)
for r in exp1_results:
    nc_max = np.max(r['nc'])
    qc_max = np.max(r['qc'])
    qr_final = r['qr'][-1]
    # Rain onset: first time qr > 0.01 g/kg
    rain_idx = np.where(r['qr'] > 0.01)[0]
    rain_onset = f"{r['time'][rain_idx[0]]:.0f} s" if len(rain_idx) > 0 else "No rain"
    print(f"{r['label']:12s} {nc_max:10.1f} {qc_max:10.4f} {qr_final:10.4f} {rain_onset:>12s}")

### Discussion — Experiment 1

1. **Twomey Effect**: More aerosols → more CCN → more but smaller droplets → higher cloud reflectivity but *suppressed* collision-coalescence → delayed or no rain.

2. **Water per droplet**: Maritime has fewer droplets sharing the same water supply → each droplet grows larger → faster collision onset.

3. **DSD width**: Continental aerosols produce a narrow DSD because the condensation growth rate $dr/dt \propto 1/r$ narrows the distribution, and the small drops struggle to cross the collision-coalescence size gap (~20 µm).

4. **Arctic**: Ultra-clean → very few but very large drops → rapid drizzle formation despite low total water content.

**Fill in your results**: Does your simulation confirm or contradict your hypothesis? Explain any surprises.

## Experiment 2: Ablation Lab — Isolating Physical Processes

### Background
The Ablation Lab lets you toggle individual physics terms on/off:
- **Kelvin effect** (curvature): surface tension increases vapor pressure over small drops
- **Raoult effect** (solute): dissolved material decreases vapor pressure
- **Collision efficiency**: Hall (1980) vs constant E = 1
- **Terminal velocity**: Beard (1976) vs simplified Stokes drag

### Hypothesis
1. Which effect matters more for condensation: Kelvin or Raoult?
2. Does setting E = 1 (perfect collection) dramatically speed up rain?
3. Is the Stokes approximation adequate for droplets < 50 µm?

In [ ]:
# Baseline parameters (Maritime)
base = {
    'n_ptcl': 1000, 'nt': 2400,
    'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
    'do_condensation': True, 'do_collision': True,
}

ablation_runs = [
    {**base, 'label': 'Baseline (all physics)'},
    {**base, 'label': 'No Kelvin', 'switch_kelvin': False},
    {**base, 'label': 'No Raoult', 'switch_solute': False},
    {**base, 'label': 'E = 1 (perfect coll.)', 'switch_E_constant': True},
    {**base, 'label': 'Stokes v_t', 'switch_vt_simple': True},
]

In [ ]:
%%time
exp2_results = []
for params in ablation_runs:
    print(f"Running {params['label']}...")
    exp2_results.append(run_simulation(params))
print("All ablation runs complete!")

In [ ]:
# Condensation ablation: Kelvin vs Raoult
plot_comparison(
    [exp2_results[0], exp2_results[1], exp2_results[2]],
    'Ablation: Condensation Physics (Kelvin & Raoult)')

In [ ]:
# Collision ablation: E and v_t
plot_comparison(
    [exp2_results[0], exp2_results[3], exp2_results[4]],
    'Ablation: Collision Physics (Efficiency & Terminal Velocity)')

In [ ]:
# DSD at final time — all ablation cases
fig, ax = plt.subplots(figsize=(10, 6))
colors_ab = [plt.cm.tab10(i) for i in range(5)]

for r, col in zip(exp2_results, colors_ab):
    nt_r = len(r['time']) - 1
    ax.plot(rm_spec * 1e6, r['spectra'][nt_r] / 1e6, color=col, lw=2, label=r['label'])

ax.set_xscale('log')
ax.set_xlabel('Radius (µm)', fontsize=12)
ax.set_ylabel('dN/dlogr (cm⁻³)', fontsize=12)
ax.set_title('Final DSD — Ablation Lab', fontsize=13)
ax.axvline(25, color='gray', ls='--', alpha=0.5, label='Rain threshold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Ablation Lab results summary
print(f"{'Case':25s} {'q_c max':>10s} {'q_r final':>10s} {'N_c max':>10s} {'Rain onset':>12s}")
print("-" * 75)
for r in exp2_results:
    qc_max = np.max(r['qc'])
    qr_final = r['qr'][-1]
    nc_max = np.max(r['nc'])
    rain_idx = np.where(r['qr'] > 0.01)[0]
    rain_onset = f"{r['time'][rain_idx[0]]:.0f} s" if len(rain_idx) > 0 else "No rain"
    print(f"{r['label']:25s} {qc_max:10.4f} {qr_final:10.4f} {nc_max:10.1f} {rain_onset:>12s}")

### Discussion — Experiment 2

1. **Kelvin effect**: Primarily important for haze particles (r < 1 µm). Removing it slightly changes activation dynamics but has little effect on mature cloud droplets.

2. **Raoult effect**: Critical for CCN activation. Without solute lowering, fewer particles activate and the DSD shifts.

3. **E = 1**: Removing the collision efficiency bottleneck allows very small droplets to collide, dramatically speeding up rain formation. This demonstrates that the "size gap" around 20-50 µm (where E is very small) is a key bottleneck.

4. **Stokes vs Beard**: For cloud droplets (< 50 µm), Stokes is a reasonable approximation. For rain drops, Stokes overestimates terminal velocity because it doesn't account for drag coefficient changes.

**Which physics term has the largest impact on rain formation?**

## Experiment 3: Updraft Velocity Sensitivity

### Background
The updraft velocity $w$ controls how quickly the parcel cools, which determines:
- The rate of supersaturation generation
- The maximum supersaturation $S_{\max}$
- The number of activated CCN

Typical values: stratocumulus ~0.3 m/s, cumulus ~1 m/s, deep convection ~3-10 m/s.

### Hypothesis
1. How does $S_{\max}$ scale with $w$?
2. Does doubling $w$ double the number of activated droplets?
3. Which $w$ value produces rain fastest?

In [ ]:
# Updraft velocity sensitivity
w_values = [0.5, 1.0, 2.0, 3.0]
exp3_params = [{
    'label': f'w = {w} m/s',
    'w': w, 'n_ptcl': 1000, 'nt': 2400,
    'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
    'do_condensation': True, 'do_collision': True,
} for w in w_values]

In [ ]:
%%time
exp3_results = []
for params in exp3_params:
    print(f"Running {params['label']}...")
    exp3_results.append(run_simulation(params))
print("Done!")

In [ ]:
plot_comparison(exp3_results, 'Experiment 3: Updraft Velocity Sensitivity')

In [ ]:
# Extract S_max and N_activated vs w
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

S_max_list = []
Nc_max_list = []
for r in exp3_results:
    S_arr = (r['RH'] - 1.0) * 100  # percent
    S_max_list.append(np.max(S_arr))
    Nc_max_list.append(np.max(r['nc']))

ax1.plot(w_values, S_max_list, 'bo-', lw=2, ms=8)
ax1.set_xlabel('Updraft Velocity w (m/s)', fontsize=12)
ax1.set_ylabel('$S_{max}$ (%)', fontsize=12)
ax1.set_title('Maximum Supersaturation vs Updraft')
ax1.grid(alpha=0.3)

ax2.plot(w_values, Nc_max_list, 'ro-', lw=2, ms=8)
ax2.set_xlabel('Updraft Velocity w (m/s)', fontsize=12)
ax2.set_ylabel('$N_c$ max (cm⁻³)', fontsize=12)
ax2.set_title('Maximum CDNC vs Updraft')
ax2.grid(alpha=0.3)

fig.suptitle('Twomey Relationship: Updraft → Supersaturation → CDNC', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Updraft sensitivity results
print(f"{'w (m/s)':>10s} {'S_max (%)':>10s} {'Nc max':>10s} {'qc max':>10s} {'qr final':>10s} {'Rain onset':>12s}")
print("-" * 65)
for w_val, r in zip(w_values, exp3_results):
    S_max = np.max((r['RH'] - 1.0) * 100)
    nc_max = np.max(r['nc'])
    qc_max = np.max(r['qc'])
    qr_final = r['qr'][-1]
    rain_idx = np.where(r['qr'] > 0.01)[0]
    rain_onset = f"{r['time'][rain_idx[0]]:.0f} s" if len(rain_idx) > 0 else "No rain"
    print(f"{w_val:10.1f} {S_max:10.4f} {nc_max:10.1f} {qc_max:10.4f} {qr_final:10.4f} {rain_onset:>12s}")

### Discussion — Experiment 3

1. **$S_{\max}$ scaling**: Higher $w$ generates supersaturation faster (source term $\propto w$), leading to higher $S_{\max}$.

2. **Twomey effect**: More supersaturation activates more CCN, but the relationship is sub-linear because the condensation sink also increases with more droplets.

3. **Rain formation paradox**: Stronger updrafts activate more CCN → more numerous but smaller droplets → *slower* collision-coalescence onset. However, stronger updrafts also mean more total water → eventually rain can still form if sufficient time/height is available.

4. **Cloud types**: Low $w$ (stratocumulus) → few large droplets → efficient drizzle. High $w$ (deep convection) → many small droplets → delayed warm rain, ice processes become important.

## Experiment 4: Superdroplet Convergence Test

### Background
The superdroplet method approximates the continuous droplet population with $N_s$ computational particles. Key questions:
- How many superdroplets are needed for reliable results?
- How much does stochastic noise (different random seeds) affect the outcome?
- What is the cost-accuracy tradeoff?

### Hypothesis
1. At what $N_s$ do results converge (< 5% difference from double)?
2. Which variable is most sensitive to $N_s$: $q_c$, $q_r$, or $N_c$?
3. How does Monte Carlo variance scale with $N_s$?

In [ ]:
# Convergence test: varying number of superdroplets
n_ptcl_values = [200, 500, 1000, 2000, 5000]
base_conv = {
    'nt': 2400,
    'N': [100.0, 20.0], 'mu': [0.050, 0.250], 'sigma': [2.0, 1.8], 'kappa': [1.0, 1.2],
    'do_condensation': True, 'do_collision': True,
}

In [ ]:
%%time
# Single seed for each n_ptcl
conv_results = []
for n_p in n_ptcl_values:
    print(f"Running n_ptcl = {n_p}...")
    conv_results.append(run_simulation({**base_conv, 'label': f'$N_s$ = {n_p}', 'n_ptcl': n_p}, seed=42))
print("Convergence test complete!")

In [ ]:
plot_comparison(conv_results, 'Experiment 4: Superdroplet Convergence')

In [ ]:
%%time
# Monte Carlo ensemble: 5 seeds for n_ptcl = 1000
seeds = [42, 123, 456, 789, 1024]
mc_results = []
for s in seeds:
    mc_results.append(run_simulation({**base_conv, 'label': f'seed={s}', 'n_ptcl': 1000}, seed=s))
print("Monte Carlo ensemble complete!")

In [ ]:
# Monte Carlo spread: mean ± std envelope
t_mc = mc_results[0]['time']
qc_all = np.array([r['qc'] for r in mc_results])
qr_all = np.array([r['qr'] for r in mc_results])

qc_mean, qc_std = np.mean(qc_all, axis=0), np.std(qc_all, axis=0)
qr_mean, qr_std = np.mean(qr_all, axis=0), np.std(qr_all, axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.fill_between(t_mc, qc_mean - qc_std, qc_mean + qc_std, alpha=0.3, color='blue')
ax1.plot(t_mc, qc_mean, 'b-', lw=2, label='Mean $q_c$')
for r in mc_results:
    ax1.plot(t_mc, r['qc'], 'b-', alpha=0.15, lw=0.5)
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('$q_c$ (g/kg)')
ax1.set_title('Cloud Water — Monte Carlo Spread ($N_s$ = 1000)')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.fill_between(t_mc, qr_mean - qr_std, qr_mean + qr_std, alpha=0.3, color='red')
ax2.plot(t_mc, qr_mean, 'r-', lw=2, label='Mean $q_r$')
for r in mc_results:
    ax2.plot(t_mc, r['qr'], 'r-', alpha=0.15, lw=0.5)
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('$q_r$ (g/kg)')
ax2.set_title('Rain Water — Monte Carlo Spread ($N_s$ = 1000)')
ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle('Monte Carlo Variance (5 seeds, $N_s$ = 1000)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Convergence metric using PEAK values (final qc ~ 0 due to rain conversion)
print("Convergence metric: ε(N) = |X(N) - X(N+1)| / X(N+1)")
print("Using peak qc and final qr (more robust than final qc which → 0 with rain)")
print("-" * 65)
print(f"{'N_s pair':>20s} {'ε(peak qc)':>12s} {'ε(final qr)':>12s} {'ε(total LWC)':>14s}")
print("-" * 65)
for i in range(len(n_ptcl_values) - 1):
    # Peak cloud water (before rain depletes it)
    qc_N = np.max(conv_results[i]['qc'])
    qc_2N = np.max(conv_results[i+1]['qc'])
    eps_qc = abs(qc_N - qc_2N) / max(abs(qc_2N), 1e-10)
    # Final rain water
    qr_N = conv_results[i]['qr'][-1]
    qr_2N = conv_results[i+1]['qr'][-1]
    eps_qr = abs(qr_N - qr_2N) / max(abs(qr_2N), 1e-10)
    # Total LWC at final time
    lwc_N = conv_results[i]['qc'][-1] + conv_results[i]['qr'][-1]
    lwc_2N = conv_results[i+1]['qc'][-1] + conv_results[i+1]['qr'][-1]
    eps_lwc = abs(lwc_N - lwc_2N) / max(abs(lwc_2N), 1e-10)
    print(f"  {n_ptcl_values[i]:5d} vs {n_ptcl_values[i+1]:5d}   {eps_qc:10.1%}   {eps_qr:10.1%}   {eps_lwc:12.1%}")

### Discussion — Experiment 4

1. **Convergence**: Condensation converges quickly (~500 superdroplets sufficient). Collision-coalescence requires more (~1000-5000) because it's a stochastic process with $O(N^2)$ pair interactions.

2. **Monte Carlo variance**: Rain water $q_r$ has higher relative variance than cloud water $q_c$ because rain formation involves a cascade of rare collision events.

3. **Cost-accuracy tradeoff**: Wall-clock time scales roughly as $O(N_s)$ for condensation and $O(N_s)$ for collision (linear due to LSM). Doubling $N_s$ approximately doubles compute time.

4. **Recommendation**: For qualitative exploration, $N_s$ = 500–1000. For quantitative results, $N_s$ = 5000+ with ensemble averaging.

## Experiment 5: Turbulent Collision Kernel

### Background
In turbulent environments, droplet collisions are enhanced by:
- **Turbulent relative velocity**: turbulent motions increase the approach speed of droplet pairs
- **Preferential concentration**: inertial particles cluster in regions of high strain (low vorticity)
- **Enhanced collision efficiency**: turbulence increases the collision cross-section

The Wang-Ayala kernel (Ayala et al. 2008; Wang & Grabowski 2009) parameterizes these effects using the TKE dissipation rate $\varepsilon$:
- $\varepsilon \approx 0.01$ m²/s³ for stratocumulus
- $\varepsilon \approx 0.04$ m²/s³ for deep convection

### Hypothesis
1. Does turbulence speed up rain onset? By how much?
2. Which droplet size range benefits most from turbulent enhancement?
3. How sensitive is the result to $\varepsilon$?

In [ ]:
# Turbulent kernel experiments
turb_runs = [
    {**base, 'label': 'Gravitational only', 'n_ptcl': 1000, 'nt': 2400},
    {**base, 'label': 'ε = 0.01 m²/s³', 'n_ptcl': 1000, 'nt': 2400,
     'switch_turb_kernel': True, 'epsilon_turb': 0.01},
    {**base, 'label': 'ε = 0.04 m²/s³', 'n_ptcl': 1000, 'nt': 2400,
     'switch_turb_kernel': True, 'epsilon_turb': 0.04},
]

In [ ]:
%%time
exp5_results = []
for params in turb_runs:
    print(f"Running {params['label']}...")
    exp5_results.append(run_simulation(params))
print("Turbulence experiments complete!")

In [ ]:
plot_comparison(exp5_results, 'Experiment 5: Turbulent Collision Kernel')

In [ ]:
# Rain onset time vs epsilon
fig, ax = plt.subplots(figsize=(8, 5))

eps_vals = [0.0, 0.01, 0.04]
onset_times = []
for r in exp5_results:
    rain_idx = np.where(r['qr'] > 0.01)[0]
    if len(rain_idx) > 0:
        onset_times.append(r['time'][rain_idx[0]])
    else:
        onset_times.append(np.nan)

ax.plot(eps_vals, onset_times, 'ro-', lw=2, ms=10)
ax.set_xlabel('$\\varepsilon$ (m²/s³)', fontsize=12)
ax.set_ylabel('Rain Onset Time (s)', fontsize=12)
ax.set_title('Effect of Turbulence on Rain Onset', fontsize=13)
ax.grid(alpha=0.3)
for i, (e, t_onset) in enumerate(zip(eps_vals, onset_times)):
    if not np.isnan(t_onset):
        ax.annotate(f'{t_onset:.0f} s', (e, t_onset), textcoords="offset points",
                    xytext=(10, 10), fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Kernel ratio: turbulent / gravitational
r_turb = np.logspace(np.log10(10e-6), np.log10(200e-6), 40)
T_tk, P_tk = 283.15, 90000.0
rho_tk = P_tk / (r_a * T_tk)
epsilon_map = 0.04

K_grav = np.zeros((len(r_turb), len(r_turb)))
K_turb_map = np.zeros((len(r_turb), len(r_turb)))

for i, r1 in enumerate(r_turb):
    for j, r2 in enumerate(r_turb):
        vt1 = ws_drops_beard(r1, rho_tk, rho_liq, P_tk, T_tk)
        vt2 = ws_drops_beard(r2, rho_tk, rho_liq, P_tk, T_tk)
        v_r = abs(vt1 - vt2)
        E_c = E_H80(r1, r2)
        
        K_grav[i,j] = np.pi * (r1 + r2)**2 * v_r * E_c
        
        urms = 2.02 * (epsilon_map / 0.04)**(1/3)
        tke_est = 1.5 * urms**2
        K_turb_map[i,j] = E_c * gck(r1, r2, vt1, vt2, epsilon_map, tke_est) * E_turb(r1, r2, epsilon_map)

# Ratio
ratio = np.where(K_grav > 1e-30, K_turb_map / K_grav, 1.0)

fig, ax = plt.subplots(figsize=(8, 7))
pcm = ax.pcolormesh(r_turb * 1e6, r_turb * 1e6, ratio.T,
                     cmap='RdYlBu_r', vmin=0.5, vmax=5, shading='auto')
plt.colorbar(pcm, ax=ax, label='$K_{turb} / K_{grav}$')
ax.set_xlabel('$r_1$ (µm)'); ax.set_ylabel('$r_2$ (µm)')
ax.set_title(f'Turbulent Kernel Enhancement ($\\varepsilon$ = {epsilon_map} m²/s³)', fontsize=13)
ax.set_xscale('log'); ax.set_yscale('log')
plt.tight_layout(); plt.show()

In [ ]:
# E_turb enhancement factor as function of collector radius
r_collector_range = np.logspace(np.log10(10e-6), np.log10(100e-6), 50)
ratios_02 = [E_turb(rc, 0.2 * rc, 0.01) for rc in r_collector_range]
ratios_05 = [E_turb(rc, 0.5 * rc, 0.01) for rc in r_collector_range]
ratios_08 = [E_turb(rc, 0.8 * rc, 0.01) for rc in r_collector_range]

ratios_02_h = [E_turb(rc, 0.2 * rc, 0.04) for rc in r_collector_range]
ratios_05_h = [E_turb(rc, 0.5 * rc, 0.04) for rc in r_collector_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(r_collector_range * 1e6, ratios_02, 'b-', label='$r_2/r_1$ = 0.2')
ax1.plot(r_collector_range * 1e6, ratios_05, 'g-', label='$r_2/r_1$ = 0.5')
ax1.plot(r_collector_range * 1e6, ratios_08, 'r-', label='$r_2/r_1$ = 0.8')
ax1.set_xlabel('Collector Radius (µm)'); ax1.set_ylabel('$E_{turb}$ Enhancement')
ax1.set_title('$\\varepsilon$ = 0.01 m²/s³ (stratocumulus)'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(r_collector_range * 1e6, ratios_02_h, 'b-', label='$r_2/r_1$ = 0.2')
ax2.plot(r_collector_range * 1e6, ratios_05_h, 'g-', label='$r_2/r_1$ = 0.5')
ax2.set_xlabel('Collector Radius (µm)'); ax2.set_ylabel('$E_{turb}$ Enhancement')
ax2.set_title('$\\varepsilon$ = 0.04 m²/s³ (deep convection)'); ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle('Turbulent Collision Efficiency Enhancement', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### Discussion — Experiment 5

1. **Turbulence accelerates rain**: The Wang-Ayala kernel enhances collision rates, especially for small droplets (10-30 µm) where the gravitational kernel is weak. This helps bridge the "size gap."

2. **Stokes number dependence**: Enhancement is strongest for droplets with Stokes number $St \approx 1$ (particle relaxation time ≈ Kolmogorov time). For $\varepsilon = 0.04$ m²/s³, this corresponds to $r \approx 20-40$ µm.

3. **Stratocumulus vs deep convection**: Higher $\varepsilon$ gives stronger enhancement. In stratocumulus ($\varepsilon \approx 0.01$), the effect is moderate. In deep convection ($\varepsilon \approx 0.04$), it can reduce rain onset time significantly.

4. **Limitations**: The lookup table approach is approximate. Real turbulence is intermittent and three-dimensional, effects not captured by mean-field parameterizations.

## Summary of Key Findings

| Experiment | Key Result | Physical Insight |
|-----------|-----------|------------------|
| 1. Aerosol Environments | Maritime rains fastest, continental suppressed | Twomey effect: more CCN → smaller drops → less collision |
| 2. Ablation Lab | E=1 has largest impact on rain | Collision efficiency bottleneck at 20-50 µm |
| 3. Updraft Sensitivity | Higher w → more CCN, delayed rain | $S_{max} \propto w^{\sim 0.5}$ (sub-linear) |
| 4. Convergence | $N_s$ ≥ 1000 for collision, ensemble for variance | Monte Carlo noise largest in $q_r$ |
| 5. Turbulence | $\varepsilon$ = 0.04 speeds up rain onset | Enhancement strongest near autoconversion size gap |

### Further Exploration Ideas

1. **Mixed presets**: What if maritime air picks up continental pollution? Create a "polluted maritime" case.
2. **Giant CCN**: Add a coarse mode with very large aerosols (r > 1 µm). How do they affect rain?
3. **Oscillating updraft**: Use `ascending_mode='sine'` to simulate cloud cycling. Does rain survive the downdraft?
4. **Sensitivity to $\kappa$**: Vary hygroscopicity from 0.1 (dust) to 1.5 (sea salt). How much does it matter?
5. **Combining turbulence + continental**: Can turbulence overcome the CCN-suppressed rain effect?

---

*End of Part 2. Congratulations on completing the PyLCM numerical experiments!*